# Week 3, Day 5 — Synthetic Data Generator

**Assignment brief:**
- Write models that can generate datasets
- Use a variety of models and prompts for diverse outputs
- Create a Gradio UI for your product

**Goal:** a general-purpose synthetic dataset generator. Any project describes what it needs—a business domain, the columns/fields, the format, how many rows—and this tool produces a usable dataset, letting you pick between several open-source and frontier models to compare outputs.

**Run on Google Colab** with a T4 GPU runtime if you want to use the open-source model paths (Runtime > Change runtime type > T4 GPU). Frontier models work fine on CPU too.

**Secrets needed** (Colab: key icon in the left sidebar; locally: environment variables):
- `HF_TOKEN` — to download open-source models from Hugging Face (accept each model's license on huggingface.co first)
- `OPENAI_API_KEY` — optional, for the frontier GPT path
- `ANTHROPIC_API_KEY` — optional, for the frontier Claude path

## 1. Install dependencies

In [ ]:
!pip install -q --upgrade gradio openai anthropic huggingface_hub
# Only needed for the open-source model paths:
!pip install -q --upgrade torch transformers accelerate bitsandbytes

## 2. Imports and sign-in

In [ ]:
import os
import io
import csv
import json
import re
import tempfile
from openai import OpenAI
import gradio as gr

IN_COLAB = "google.colab" in str(get_ipython())

if IN_COLAB:
    from google.colab import userdata
    def get_secret(name):
        try:
            return userdata.get(name)
        except Exception:
            return None
else:
    def get_secret(name):
        return os.environ.get(name)

hf_token = get_secret('HF_TOKEN')
openai_api_key = get_secret('OPENAI_API_KEY')
anthropic_api_key = get_secret('ANTHROPIC_API_KEY')

if hf_token:
    from huggingface_hub import login
    login(hf_token, add_to_git_credential=True)

openai_client = OpenAI(api_key=openai_api_key) if openai_api_key else None

print("HF token loaded:", bool(hf_token))
print("OpenAI key loaded:", bool(openai_api_key))
print("Anthropic key loaded:", bool(anthropic_api_key))

## 3. Model registry

This is the "variety of models" part of the brief — a mix of open-source (run locally on the GPU) and frontier (via API) models, all selectable from the same UI.

In [ ]:
OPEN_SOURCE_MODELS = {
    "Llama 3.1 8B": "meta-llama/Meta-Llama-3.1-8B-Instruct",
    "Qwen2 7B": "Qwen/Qwen2-7B-Instruct",
    "Phi-3 Mini": "microsoft/Phi-3-mini-4k-instruct",
}

FRONTIER_MODELS = {
    "GPT-4o-mini": "gpt-4o-mini",
    "Claude Sonnet": "claude-sonnet-4-6",
}

ALL_MODEL_CHOICES = list(FRONTIER_MODELS.keys()) + list(OPEN_SOURCE_MODELS.keys())

## 4. Prompt construction

One general system prompt plus a user prompt built from whatever the project tells us: business/domain description, desired columns (optional — the model infers sensible ones if left blank), output format, row count, and a free-text "style" hint that gives you a second lever (besides model choice) for varying the output.

In [ ]:
SYSTEM_PROMPT = (
    "You are an expert synthetic data generator used as a backend tool by many different projects. "
    "Given a description of a business or domain, you generate a realistic, internally consistent dataset. "
    "Follow the requested format exactly and output ONLY the data — no commentary, no markdown code fences, "
    "no explanations before or after. If specific columns aren't given, infer sensible ones for the domain. "
    "Vary values realistically (no obviously repeated or templated rows)."
)

def build_user_prompt(description: str, columns: str, data_format: str, num_records: int, style_hint: str) -> str:
    parts = [f"Business/domain: {description.strip()}"]
    if columns.strip():
        parts.append(f"Required columns/fields: {columns.strip()}")
    parts.append(f"Output format: {data_format}")
    parts.append(f"Number of records: {num_records}")
    if style_hint.strip():
        parts.append(f"Style/diversity guidance: {style_hint.strip()}")

    if data_format == "CSV":
        parts.append("Output raw CSV: a header row, then the data rows. No extra text.")
    elif data_format == "JSON":
        parts.append("Output a raw JSON array of objects, one object per record. No extra text.")
    else:
        parts.append("Output a clean plain-text table with aligned columns. No extra text.")

    return "\n".join(parts)

## 5. Generation — open-source path

Models are loaded lazily and cached per session, so switching between them the first time is slow but re-using one is fast.

In [ ]:
_loaded_models = {}  # model_id -> (tokenizer, model)

def _load_open_source_model(model_id: str):
    if model_id in _loaded_models:
        return _loaded_models[model_id]
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4"
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id, device_map="auto", quantization_config=quant_config, trust_remote_code=True
    )
    _loaded_models[model_id] = (tokenizer, model)
    return tokenizer, model

def generate_open_source(model_id: str, system_prompt: str, user_prompt: str, temperature: float) -> str:
    tokenizer, model = _load_open_source_model(model_id)
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True
    ).to(model.device)
    outputs = model.generate(
        inputs, max_new_tokens=3000,
        do_sample=True, temperature=max(temperature, 0.01),
        pad_token_id=tokenizer.eos_token_id
    )
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full_text.split(user_prompt)[-1].strip()

## 6. Generation — frontier path

In [ ]:
def generate_frontier(model_key: str, system_prompt: str, user_prompt: str, temperature: float) -> str:
    if model_key == "Claude Sonnet":
        import anthropic
        client = anthropic.Anthropic(api_key=anthropic_api_key)
        response = client.messages.create(
            model=FRONTIER_MODELS[model_key],
            max_tokens=3000,
            temperature=temperature,
            system=system_prompt,
            messages=[{"role": "user", "content": user_prompt}]
        )
        return response.content[0].text
    else:
        response = openai_client.chat.completions.create(
            model=FRONTIER_MODELS[model_key],
            temperature=temperature,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]
        )
        return response.choices[0].message.content

## 7. Cleanup + save to a real file

Models sometimes wrap output in code fences even when told not to — strip those, then write an actual `.csv` / `.json` / `.txt` file so the dataset is immediately usable by whatever project asked for it, not just text in a chat window.

In [ ]:
def strip_code_fences(text: str) -> str:
    text = text.strip()
    text = re.sub(r"^```[a-zA-Z]*\n", "", text)
    text = re.sub(r"\n```$", "", text)
    return text.strip()

def save_dataset(raw_text: str, data_format: str) -> str:
    cleaned = strip_code_fences(raw_text)
    ext = {"CSV": "csv", "JSON": "json", "Tabular": "txt"}[data_format]
    fd, path = tempfile.mkstemp(suffix=f".{ext}", prefix="synthetic_dataset_")
    with os.fdopen(fd, "w") as f:
        f.write(cleaned)
    return path, cleaned

## 8. Top-level generator function

In [ ]:
def generate_dataset(description, columns, data_format, num_records, style_hint, model_choice, temperature):
    if not description.strip():
        return "Please describe the business/domain first.", None

    user_prompt = build_user_prompt(description, columns, data_format, num_records, style_hint)

    try:
        if model_choice in FRONTIER_MODELS:
            raw = generate_frontier(model_choice, SYSTEM_PROMPT, user_prompt, temperature)
        else:
            raw = generate_open_source(OPEN_SOURCE_MODELS[model_choice], SYSTEM_PROMPT, user_prompt, temperature)
    except Exception as e:
        return f"Error generating dataset: {e}", None

    path, cleaned = save_dataset(raw, data_format)
    return cleaned, path

## 9. Gradio UI

In [ ]:
with gr.Blocks(title="Synthetic Data Generator") as demo:
    gr.Markdown(
        "# 🧪 Synthetic Data Generator\n"
        "Describe what your project needs, pick a model, and get a ready-to-use dataset."
    )
    with gr.Row():
        with gr.Column(scale=1):
            description = gr.Textbox(
                label="Business / domain description",
                placeholder="e.g. A subscription box service for artisanal coffee, selling to US customers",
                lines=4
            )
            columns = gr.Textbox(
                label="Required columns (optional — leave blank to let the model choose)",
                placeholder="e.g. customer_id, plan, monthly_price, signup_date, region",
                lines=2
            )
            style_hint = gr.Textbox(
                label="Style / diversity guidance (optional)",
                placeholder="e.g. include some edge cases like cancelled subscriptions and refunds",
                lines=2
            )
            with gr.Row():
                data_format = gr.Dropdown(["CSV", "JSON", "Tabular"], value="CSV", label="Format")
                num_records = gr.Slider(5, 200, value=25, step=5, label="Number of records")
            with gr.Row():
                model_choice = gr.Dropdown(ALL_MODEL_CHOICES, value=ALL_MODEL_CHOICES[0], label="Model")
                temperature = gr.Slider(0.0, 1.5, value=0.8, step=0.1, label="Temperature (diversity)")
            generate_btn = gr.Button("Generate dataset", variant="primary")

        with gr.Column(scale=1):
            output_preview = gr.Textbox(label="Preview", lines=25)
            output_file = gr.File(label="Download dataset")

    generate_btn.click(
        fn=generate_dataset,
        inputs=[description, columns, data_format, num_records, style_hint, model_choice, temperature],
        outputs=[output_preview, output_file]
    )

demo.launch(debug=True)

## Notes / ways to extend this

- **"Used by any project"**: the description/columns/format fields are the whole interface — nothing here is hardcoded to one domain, so any team can plug in their own spec.
- **Diverse outputs**: two independent levers — model choice (different models have different "styles") and temperature (higher = more varied/creative rows). Try the same description across a couple of models and compare.
- **Batch generation**: call `generate_dataset` in a loop with different `style_hint`s or seeds and concatenate the results if you need more rows than one call comfortably produces.
- **Validation**: for CSV/JSON, you could parse the cleaned output with `csv.reader` / `json.loads` and re-prompt the model if parsing fails, for a more robust pipeline.
- **Programmatic use**: since `generate_dataset(...)` is a plain Python function, another notebook or script can `import` this file's logic directly instead of going through the UI — that's what makes it reusable "by any project."